In [7]:
import os
import re
import time
import warnings

from google.adk.agents.llm_agent import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions.in_memory_session_service import (
    InMemorySessionService,
)
# [CHANGE 1] Import ADK built-in Google Search tool (Instruction 3.2)
from google.adk.tools import google_search
from google.genai import types
from IPython.display import clear_output, display as ipy_display
import ipywidgets as widgets
import requests
import vertexai

# REQUIRED FIX — disable LiteLLM MCP subsystem
os.environ["LITELLM_DISABLE_MCP"] = "true"

# Suppress experimental JSON schema warning
warnings.filterwarnings(
    "ignore",
    message=".*JSON_SCHEMA_FOR_FUNC_DECL is enabled.*"
)

# ------------------------------------------------------------
# 1. INITIALIZATION & SETUP
# ------------------------------------------------------------

PROJECT_ID = !gcloud config get-value project
PROJECT_ID = PROJECT_ID[0]
LOCATION = "us-central1"
MODEL_LOCATION = "global"
STAGING_BUCKET = "gs://agent-engine-staging-bucket-dar2"

# Define project information manually if the above code didn't work
if PROJECT_ID == "(unset)" or not PROJECT_ID:
    PROJECT_ID = "qwiklabs-gcp-02-c413994bc4ba"

print(f"Project ID: {PROJECT_ID}")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_QUOTA_PROJECT"] = PROJECT_ID
# [CHANGE 2] Enable Vertex AI backend for native ADK Gemini & tools
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["ADK_SUPPRESS_GEMINI_LITELLM_WARNINGS"] = "true"

vertexai.init(
    project=PROJECT_ID,
    location=MODEL_LOCATION,
    staging_bucket=STAGING_BUCKET,
)

session_service = InMemorySessionService()

# Lists to keep logs programmatically as requested
prompt_logs = []
response_logs = []

# ------------------------------------------------------------
# CALLBACKS FOR LOGGING & VALIDATION
# ------------------------------------------------------------


def validate_user_input(session, message):
    """Callback to validate user input before sending to model."""
    if not message or not message.parts:
        raise ValueError("Empty message encountered.")

    user_text = " ".join(
        [part.text for part in message.parts if part.text]
    ).strip()

    if len(user_text) < 2:
        raise ValueError(
            "Query is too short. Please provide a valid request."
        )

    # Security / Anti-Injection Validation
    malicious_patterns = [
        r"(?i)system\s+prompt",
        r"(?i)ignore\s+previous",
        r"(?i)delete\s+from",
        r"(?i)drop\s+table",
        r"(?i)select\s+\*\s+from",
    ]
    for pattern in malicious_patterns:
        if re.search(pattern, user_text):
            raise ValueError(
                "Security violation: Input contains prohibited commands "
                "or prompt injection."
            )

    # US Location Check Validation (applies when asking for weather)
    non_us_indicators = [
        "london", "paris", "tokyo", "berlin", "beijing", "sydney", "rome",
        "canada", "mexico", "france", "germany", "uk", "united kingdom",
        "japan", "australia",
    ]
    normalized_text = user_text.lower()
    is_weather_query = any(
        w in normalized_text for w in ["weather", "forecast", "temperature"]
    )
    for indicator in non_us_indicators:
        if (
            f", {indicator}" in normalized_text
            or normalized_text == indicator
            or (is_weather_query and indicator in normalized_text)
        ):
            raise ValueError(
                f"NWS Forecast only supports United States locations. "
                f"'{user_text}' is recognized as outside the US."
            )

    return message


def log_user_prompt(session, message):
    """Callback to log user prompts."""
    user_text = " ".join(
        [part.text for part in message.parts if part.text]
    ).strip()
    log_entry = f"[Session: {session.session_id}] Prompt: {user_text}"
    prompt_logs.append(log_entry)
    print(f"[CALLBACK LOG - USER PROMPT]: {user_text}")


def log_model_response(session, message):
    """Callback to log model responses."""
    model_text = " ".join(
        [part.text for part in message.parts if part.text]
    ).strip()
    log_entry = f"[Session: {session.session_id}] Response: {model_text}"
    response_logs.append(log_entry)
    print(f"[CALLBACK LOG - MODEL RESPONSE]: {model_text}")


# ------------------------------------------------------------
# 2. DEFINE TOOLS FOR WEATHER AGENT
# ------------------------------------------------------------


def geocode_place_free(address: str) -> str:
    """Geocode an address into latitude and longitude coordinates."""
    base_url = "https://nominatim.openstreetmap.org/search"
    params = {"q": address, "format": "json", "limit": 1}
    headers = {
        "User-Agent": "RussellWeatherApp/1.0 (russell.f.zimmermann@saic.com)"
    }
    try:
        response = requests.get(
            base_url, params=params, headers=headers, timeout=10
        )
        response.raise_for_status()
        data = response.json()
        if data:
            return f"{data[0]['lat']}, {data[0]['lon']}"
    except Exception:
        pass
    return "Error: Location not found."


def get_nws_forecast(latitude: float, longitude: float) -> str:
    """Retrieve the National Weather Service (NWS) forecast."""
    headers = {
        "User-Agent": "RussellWeatherApp/1.0 (russell.f.zimmermann@saic.com)"
    }
    points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
    try:
        points_response = requests.get(
            points_url, headers=headers, timeout=10
        )
        points_response.raise_for_status()
        forecast_url = points_response.json()["properties"]["forecast"]
        forecast_response = requests.get(
            forecast_url, headers=headers, timeout=10
        )
        forecast_response.raise_for_status()
        periods = forecast_response.json()["properties"]["periods"]
        formatted = []
        for period in periods[:2]:
            formatted.append(
                f"{period['name']}: {period['temperature']}°"
                f"{period['temperatureUnit']}, {period['shortForecast']}."
            )
        return "\n\n".join(formatted)
    except Exception as e:
        return f"Error: {e}"


# ------------------------------------------------------------
# 3. DEFINE MULTI-AGENT SYSTEM
#    (Requirements 2.1, 2.2 & Instructions 3.2, 3.3)
# ------------------------------------------------------------

weather_instruction = (
    "You are a helpful weather assistant. "
    "When a user provides a location, use your geocoding and forecast tools "
    "to find the weather forecast for that location. "
    "If the location is generic (like 'Chicago' or 'Lancaster'), proceed with "
    "geocoding the most probable option. "
    "IMPORTANT: Always explicitly state the identified state (or region) "
    "and country for the city you chose in your final response to the user, "
    "so they know exactly which location's forecast you retrieved (e.g., "
    "'Retrieving weather for Lancaster, Pennsylvania, United States'). "
    "If the user clarifies a specific state or country, use that precise "
    "location for geocoding."
)

# [CHANGE 3] Instruction for Search Sub-Agent
# (Requirement 2.1.3 & Instruction 3.2)
search_instruction = (
    "You are a helpful search assistant. "
    "Use the built-in google_search tool to find accurate, up-to-date "
    "information to answer user questions about general knowledge, news, "
    "events, and facts."
)

# [CHANGE 4] Instruction for Root Coordinating Agent
# (Requirements 2.1.1, 2.2 & Instruction 3.3)
root_instruction = (
    "You are the root coordinating agent. Your job is to receive user "
    "requests and delegate tasks appropriately to your sub-agents:\n"
    "- Delegate weather, temperature, and forecast questions to the "
    "weather sub-agent.\n"
    "- Delegate general knowledge, factual, current events, and search "
    "questions to the search sub-agent."
)

# --- FLASH MULTI-AGENT HIERARCHY (Default) ---
# Requirement 2.1.2: Weather Sub-Agent
weather_agent = LlmAgent(
    name="weather_agent",
    model="gemini-2.5-flash",
    description="Retrieves US weather forecasts given a city or location.",
    instruction=weather_instruction,
    tools=[geocode_place_free, get_nws_forecast],
)

# Requirement 2.1.3 & Instruction 3.2: Search Sub-Agent
search_agent = LlmAgent(
    name="search_agent",
    model="gemini-2.5-flash",
    description=(
        "Answers general knowledge, news, and search questions "
        "using Google Search."
    ),
    instruction=search_instruction,
    tools=[google_search],
)

# Requirement 2.1.1, 2.2 & Instruction 3.3: Root Agent
root_agent = LlmAgent(
    name="root_agent",
    model="gemini-2.5-flash",
    description=(
        "Root coordinating agent that delegates tasks to "
        "weather_agent and search_agent."
    ),
    instruction=root_instruction,
    sub_agents=[weather_agent, search_agent],
)

agent_flash = root_agent
runner_flash = Runner(
    agent=agent_flash,
    app_name="VertexFlashMultiAgentApp",
    session_service=session_service,
    auto_create_session=True,
)

# --- PRO MULTI-AGENT HIERARCHY ---
weather_agent_pro = LlmAgent(
    name="weather_agent_pro",
    model="gemini-2.5-pro",
    description="Retrieves US weather forecasts given a city or location.",
    instruction=weather_instruction,
    tools=[geocode_place_free, get_nws_forecast],
)

search_agent_pro = LlmAgent(
    name="search_agent_pro",
    model="gemini-2.5-pro",
    description=(
        "Answers general knowledge, news, and search questions "
        "using Google Search."
    ),
    instruction=search_instruction,
    tools=[google_search],
)

agent_pro = LlmAgent(
    name="root_agent_pro",
    model="gemini-2.5-pro",
    description=(
        "Root coordinating agent (Pro) that delegates tasks to sub-agents."
    ),
    instruction=root_instruction,
    sub_agents=[weather_agent_pro, search_agent_pro],
)

runner_pro = Runner(
    agent=agent_pro,
    app_name="VertexProMultiAgentApp",
    session_service=session_service,
    auto_create_session=True,
)

# Assign Callback Lists directly to Runner Attributes
runner_flash.pre_run_callbacks = [validate_user_input, log_user_prompt]
runner_flash.post_run_callbacks = [log_model_response]

runner_pro.pre_run_callbacks = [validate_user_input, log_user_prompt]
runner_pro.post_run_callbacks = [log_model_response]

# ------------------------------------------------------------
# 4. SIMPLIFIED INTERACTIVE CHAT INTERFACE
# ------------------------------------------------------------

session_id = "multi_agent_session"
user_id = "russell"

chat_output = widgets.Output()

model_dropdown = widgets.Dropdown(
    options=[("Gemini 2.5 Flash", "flash"), ("Gemini 2.5 Pro", "pro")],
    value="flash",
    description="Select Model",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="260px"),
)

custom_css = """
<style>
.widget-dropdown select {
    background-color: #212529 !important;
    color: #ffffff !important;
    border: 1px solid #495057 !important;
}
</style>
"""
css_widget = widgets.HTML(custom_css)

user_text_input = widgets.Text(
    placeholder=(
        "Ask for US weather or a Google Search question and press Enter..."
    ),
    layout=widgets.Layout(width="60%"),
)

send_button = widgets.Button(
    description="Send",
    button_style="info",
    layout=widgets.Layout(width="15%"),
)
exit_button = widgets.Button(
    description="End Chat",
    button_style="danger",
    layout=widgets.Layout(width="15%"),
)

top_row = widgets.HBox(
    [model_dropdown],
    layout=widgets.Layout(grid_gap="20px"),
)
bottom_row = widgets.HBox(
    [user_text_input, send_button, exit_button],
    layout=widgets.Layout(grid_gap="10px"),
)
controls = widgets.VBox(
    [top_row, bottom_row],
    layout=widgets.Layout(margin="10px 0px"),
)

interface_box = widgets.VBox([chat_output, controls])


def append_message(sender, text, is_agent=False):
    """Append a formatted message to the chat output widget."""
    with chat_output:
        color = "#0d6efd" if not is_agent else "#198754"
        sender_name = "You" if not is_agent else sender
        ipy_display(widgets.HTML(
            f"<b style='color: {color};'>{sender_name}:</b> {text}<br>"
        ))


def disable_interface():
    """Disable interactive widgets when the conversation ends."""
    model_dropdown.disabled = True
    user_text_input.disabled = True
    send_button.disabled = True
    exit_button.disabled = True
    with chat_output:
        ipy_display(widgets.HTML(
            "<hr><b style='color: #dc3545;'>Conversation ended.</b>"
        ))


def handle_submit(sender):
    """Handle user query submission and stream multi-agent events."""
    user_query = user_text_input.value.strip()
    if not user_query:
        return
    user_text_input.value = ""

    append_message(user_id, user_query, is_agent=False)
    if user_query.lower() in ["exit", "quit"]:
        disable_interface()
        return

    selected_model_key = model_dropdown.value
    active_runner = runner_pro if selected_model_key == "pro" else runner_flash
    model_display_name = (
        "Gemini 2.5 Pro" if selected_model_key == "pro"
        else "Gemini 2.5 Flash"
    )

    with chat_output:
        ipy_display(widgets.HTML(
            f"<small style='color: #6c757d; font-style: italic;'>"
            f"[Routing message to Root Agent ({model_display_name})]</small>"
        ))

    message_content = types.Content(
        role="user",
        parts=[types.Part.from_text(text=user_query)],
    )
    try:
        events = active_runner.run(
            user_id=user_id,
            session_id=session_id,
            new_message=message_content,
        )
        for event in events:
            author = getattr(event, "author", "Agent")
            actions = getattr(event, "actions", None)
            transfer_target = getattr(actions, "transfer_to_agent", None)
            if transfer_target:
                with chat_output:
                    ipy_display(widgets.HTML(
                        f"<small style='color: #6c757d;'>"
                        f"&#8627; Delegating to sub-agent: "
                        f"<b>{transfer_target}</b></small>"
                    ))
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        append_message(
                            f"Agent ({author})",
                            part.text,
                            is_agent=True,
                        )
    except Exception as e:
        append_message("System", f"Error validation: {e}", is_agent=True)


user_text_input.on_submit(handle_submit)
send_button.on_click(handle_submit)
exit_button.on_click(lambda b: disable_interface())

clear_output()
ipy_display(css_widget)
ipy_display(interface_box)

with chat_output:
    ipy_display(widgets.HTML(
        f"<b style='color: #6c757d;'>Multi-Agent System initialized "
        f"(Project: {PROJECT_ID} | "
        f"root_agent -> [weather_agent, search_agent]). "
        f"Choose your model and type a query below:</b><br>"
    ))

HTML(value='\n<style>\n.widget-dropdown select {\n    background-color: #212529 !important;\n    color: #fffff…

In [5]:
# ------------------------------------------------------------
# SECOND CODE CELL: Instructions 3.4 & 3.5
# Test agents & output events demonstrating sub-agent usage
# ------------------------------------------------------------


def test_agent_and_output_events(prompt: str, test_session_id: str):
    """Run a test prompt against root_agent and print event details."""
    print("=" * 75)
    print(f"USER REQUEST: {prompt}")
    print("=" * 75)

    user_msg = types.Content(
        role="user",
        parts=[types.Part.from_text(text=prompt)],
    )
    events = runner_flash.run(
        user_id="challenge3_tester",
        session_id=test_session_id,
        new_message=user_msg,
    )

    for idx, event in enumerate(events, start=1):
        author = getattr(event, "author", "unknown")
        print(f"\n[Event #{idx}] Author: {author}")

        # Display sub-agent transfer action
        actions = getattr(event, "actions", None)
        transfer_target = getattr(actions, "transfer_to_agent", None)
        if transfer_target:
            print(f"  -> Action (Transfer to Sub-Agent): {transfer_target}")

        # Display tool calls, tool responses, and text output
        if event.content and event.content.parts:
            for part in event.content.parts:
                if getattr(part, "function_call", None):
                    fc = part.function_call
                    args_dict = dict(fc.args) if fc.args else {}
                    print(f"  -> Tool Call: {fc.name}({args_dict})")
                if getattr(part, "function_response", None):
                    fr = part.function_response
                    print(
                        f"  -> Tool Response from `{fr.name}`: {fr.response}"
                    )
                if getattr(part, "text", None):
                    print(f"  -> Text Output: {part.text.strip()}")

        # Display Google Search grounding queries when search_agent runs
        gm = getattr(event, "grounding_metadata", None)
        queries = getattr(gm, "web_search_queries", None)
        if queries:
            print(f"  -> Google Search Queries Executed: {queries}")


# Test 1: Verify Root Agent delegates to weather_agent
test_agent_and_output_events(
    "What is the weather forecast for Austin, Texas?",
    test_session_id=f"test_weather_{int(time.time())}",
)

# Test 2: Verify Root Agent delegates to search_agent (Google Search)
test_agent_and_output_events(
    "Who won the most recent Super Bowl and what was the final score?",
    test_session_id=f"test_search_{int(time.time())}",
)

USER REQUEST: What is the weather forecast for Austin, Texas?

[Event #1] Author: root_agent
  -> Tool Call: transfer_to_agent({'agent_name': 'weather_agent'})

[Event #2] Author: root_agent
  -> Action (Transfer to Sub-Agent): weather_agent
  -> Tool Response from `transfer_to_agent`: {'result': None}

[Event #3] Author: weather_agent
  -> Tool Call: geocode_place_free({'address': 'Austin, Texas'})

[Event #4] Author: weather_agent
  -> Tool Response from `geocode_place_free`: {'result': '30.2649261, -97.7474086'}

[Event #5] Author: weather_agent
  -> Tool Call: get_nws_forecast({'latitude': 30.2649261, 'longitude': -97.7474086})

[Event #6] Author: weather_agent
  -> Tool Response from `get_nws_forecast`: {'result': 'This Afternoon: 82°F, Mostly Sunny.\n\nTonight: 64°F, Partly Cloudy.'}

[Event #7] Author: weather_agent
  -> Text Output: Retrieving weather for Austin, Texas, United States.
This Afternoon: 82°F, Mostly Sunny.

Tonight: 64°F, Partly Cloudy.
USER REQUEST: Who won the m